# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [1]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [3]:
student_id = "student_02"
handle = "B1TVChannel"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\raw\student_02_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [4]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'RonwGY_bfyqIH9zxtfhxaU_lReA',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'i-MydfuK9Moc1bCyvLBabjzBtBw',
   'id': 'UCeqNP-Wt7YNjPyH6XhMYPxw'}]}

In [5]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCeqNP-Wt7YNjPyH6XhMYPxw'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [6]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'CwUpd5BrdgxSAqmwIY5QCXlLNuI',
 'id': {'kind': 'youtube#video', 'videoId': '-NYRzPKc0Uk'},
 'snippet': {'publishedAt': '2026-05-06T15:35:49Z',
  'channelId': 'UCeqNP-Wt7YNjPyH6XhMYPxw',
  'title': 'POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuza GRINDEANU premier”',
  'description': 'S.TĂNASE: „PNL nu se va așeza la MASĂ CU PSD/ /BOLOJAN era bun să ducă la capăt MISIUNEA”.',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/-NYRzPKc0Uk/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/-NYRzPKc0Uk/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/-NYRzPKc0Uk/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'B1',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-06T15:35:49Z'}}

In [7]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': '-NYRzPKc0Uk',
  'video_title': 'POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuza GRINDEANU premier”',
  'video_date': '2026-05-06'},
 {'video_id': 'eTJWWC7A_6k',
  'video_title': 'Stelian Tănase denunță ipocrizia politicienilor care au generat criza. Vina lui Nicușor Dan._B1TV_',
  'video_date': '2026-05-06'},
 {'video_id': '3xIx29xlRfs',
  'video_title': 'POLITICA ZILEI. TRUMP oprește GHIDAREA navelor în ORMUZ/RUSIA, PRIMA PARADĂ fără Tancuri DE 9 MAI.P2',
  'video_date': '2026-05-06'},
 {'video_id': 'xO7Qvs__V0Y',
  'video_title': 'POLITICA ZILEI. SURSE: Guvern PSD-UDMR-Minorități / SURSE: USR și PNL ar susține un GUVERN MINORITAR',
  'video_date': '2026-05-06'},
 {'video_id': 'wrh2kbxmrvQ',
  'video_title': 'TALK B1. Dragoș Pîslaru, invitat: este posibil să intru în PNL. Care sunt intențiile lui BOLOJAN P3',
  'video_date': '2026-05-06'},
 {'video_id': 'EcRdjkJ78co',
  'video_title': 'Cl. MANDA a anunțat deciziile luate de PSD după ce 

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [8]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuz


C:\Users\Ionela\AppData\Local\Temp\ipykernel_21216\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Stelian Tănase denunță ipocrizia politicienilor care au generat criza. Vina lui 
Colectez: POLITICA ZILEI. TRUMP oprește GHIDAREA navelor în ORMUZ/RUSIA, PRIMA PARADĂ fără
Colectez: POLITICA ZILEI. SURSE: Guvern PSD-UDMR-Minorități / SURSE: USR și PNL ar susține
Colectez: TALK B1. Dragoș Pîslaru, invitat: este posibil să intru în PNL. Care sunt intenț
Colectez: Cl. MANDA a anunțat deciziile luate de PSD după ce Guvernul a picat: Avem pe mas
Colectez: TALK B1 cu Gabriela Mihai. Și AUR și PSD vor PREMIER./Strategul AUR, exclusiv la
Colectez: TALK B1 cu Gabriela Mihai. Noile MAJORITĂȚI. Strategii. CINE IA PUTEREA?/Consult
Colectez: Andrei ȚĂRANU, analist politic, CUM va arăta NOUL GUVERN?_Știri B1TV_6 mai 2026
Colectez: ULTIMA ORĂ: IRANUL a lovit o navă FRANCEZĂ în strâmtoarea ORMUZ_Știri B1TV_6 mai


89

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [9]:
comments[:3]

[{'id': 'yt_-NYRzPKc0Uk_UgzMYAuROQ3TATdYRbV4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'B1TVChannel',
  'text_raw': 'Si de ce nu ar fi Viata dupa grindeni si ioane,mai, mai...',
  'video_id': '-NYRzPKc0Uk',
  'video_title': 'POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuza GRINDEANU premier”',
  'video_date': '2026-05-06',
  'comment_date': '2026-05-06',
  'likes': 15,
  'collected_at': '2026-05-06'},
 {'id': 'yt_-NYRzPKc0Uk_UgxQEPjlpqg24bzoWyF4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'B1TVChannel',
  'text_raw': 'Parșivul grindeanu sa nu mai bâzâie....la munca nu la întins mana',
  'video_id': '-NYRzPKc0Uk',
  'video_title': 'POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuza GRINDEANU premier”',
  'video_date': '2026-05-06',
  'comment_date': '2026-05-06',
  'likes': 8,
  'collected_at': '2026-05-06'},
 {'id': 'yt_-NYRzPKc0Uk_UgwnFG7nqrZl18XTw4d4AaABAg',
  'source_platform': 'youtube

In [10]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [11]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [12]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_-NYRzPKc0Uk_UgzMYAuROQ3TATdYRbV4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'B1TVChannel',
 'text_raw': 'Si de ce nu ar fi Viata dupa grindeni si ioane,mai, mai...',
 'video_id': '-NYRzPKc0Uk',
 'video_title': 'POLITICA ZILEI. S.TĂNASE:„E un joc de CACEALMA ca să CÂȘTIGE TIMP/PSD n-ar refuza GRINDEANU premier”',
 'video_date': '2026-05-06',
 'comment_date': '2026-05-06',
 'likes': 15,
 'collected_at': '2026-05-06',
 'text': 'Si de ce nu ar fi Viata dupa grindeni si ioane,mai, mai...'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [13]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 89
Comentarii după filtrarea lungimii: 53


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [14]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 53


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [15]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 53


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [16]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 53
Fișier: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\cleaned\student_02_youtube_clean.jsonl


# Functia de curatare

In [17]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [18]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 89
Comentarii curate: 53


In [19]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Parșivul grindeanu sa nu mai bâzâie....la munca nu la întins mana
CLEAN: Parșivul grindeanu sa nu mai bâzâie....la munca nu la întins mana
---
RAW: Taci ca vati bătut joc de tara gunoiule a fost prim-ministru cea făcut au furat toți ca ala cu stuful
CLEAN: Taci ca vati bătut joc de tara gunoiule a fost prim-ministru cea făcut au furat toți ca ala cu stuful
---
RAW: Gustere Grindeanu  îți e frică să mergi la anticipate ca îți moare PSD ul duceți-vă
CLEAN: Gustere Grindeanu îți e frică să mergi la anticipate ca îți moare PSD ul duceți-vă
---


In [20]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5\data\cleaned\student_02_youtube_clean.jsonl
Comentarii salvate: 53


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.